# 📂 Day 143 — Streamlit: File Upload + Dynamic EDA Dashboard
**Month 8 | Streamlit + Python Deployment**

| Field | Detail |
|---|---|
| **Day** | 143 |
| **Topic** | `st.file_uploader` · Auto EDA · Dynamic Filters · Download Button |
| **Dataset** | FreelanceHub India — 500 rows, seed=141 |
| **Max Score** | 80 pts + 10★ Bonus |
| **Deliverable** | `day143_app.py` — a Streamlit app any client can upload their own CSV to |

---
> **The goal:** Build a universal CSV analyser. A client uploads any dataset → they instantly see shape, types, nulls, stats, charts, and a downloadable processed file.  
> This is one of the highest-ROI Streamlit deliverables on Upwork.

---


---
## 🔒 Section 1 — Raw Data
**Do not modify this section. Run it once to generate and save the dataset.**


In [1]:
# ── RAW DATA GENERATOR — do not modify ───────────────────────────────────────
import pandas as pd
import numpy as np

np.random.seed(141)
n = 500

categories     = ['Web Development', 'Data Analytics', 'ML/AI', 'Design',
                   'Content Writing', 'Digital Marketing']
cities         = ['Mumbai', 'Delhi', 'Bangalore', 'Hyderabad', 'Chennai', 'Pune', 'Kolkata']
levels         = ['Rising Talent', 'Level 1', 'Level 2', 'Top Rated']
statuses       = ['Completed', 'In Progress', 'Cancelled']
months         = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

df_raw = pd.DataFrame({
    'project_id'      : [f'P{str(i).zfill(3)}' for i in range(1, n+1)],
    'category'        : np.random.choice(categories, n, p=[0.25,0.20,0.15,0.15,0.15,0.10]),
    'budget_inr'      : np.random.randint(5000, 150001, n),
    'duration_days'   : np.random.randint(3, 91, n),
    'client_rating'   : np.round(np.random.uniform(2.5, 5.0, n), 1),
    'freelancer_level': np.random.choice(levels, n, p=[0.30,0.30,0.25,0.15]),
    'status'          : np.random.choice(statuses, n, p=[0.65,0.25,0.10]),
    'city'            : np.random.choice(cities, n),
    'platform_fee_pct': np.random.choice([10, 15, 20], n, p=[0.20,0.50,0.30]),
    'month'           : np.random.choice(months, n),
})
df_raw['net_earnings'] = np.round(df_raw['budget_inr'] * (1 - df_raw['platform_fee_pct']/100), 2)

# Save CSV — this is the file you will "upload" in your Streamlit app for testing
df_raw.to_csv('freelancehub_india.csv', index=False)
print(f"✅ Dataset saved: {df_raw.shape[0]} rows × {df_raw.shape[1]} cols")
print(df_raw.head(3))


✅ Dataset saved: 500 rows × 11 cols
  project_id           category  budget_inr  duration_days  client_rating  \
0       P001  Digital Marketing       70355             89            3.7   
1       P002              ML/AI       95518             13            3.9   
2       P003    Web Development      149779             76            3.0   

  freelancer_level     status     city  platform_fee_pct month  net_earnings  
0        Top Rated  Completed  Chennai                20   Feb      56284.00  
1        Top Rated  Completed     Pune                10   Dec      85966.20  
2          Level 2  Completed  Kolkata                15   Aug     127312.15  


---
## 📚 Section 2 — Concept Notes

### Why `st.file_uploader` matters for freelance work
A dashboard hardcoded to one CSV is a demo. A dashboard where the client uploads their **own** CSV is a product.  
This distinction justifies 2–3x higher Upwork rates.

---

### Core concepts today

| Concept | What it does | Why it matters |
|---|---|---|
| `st.file_uploader()` | Widget that accepts file drag-and-drop | Entry point for any dynamic app |
| `io.StringIO` | Converts uploaded bytes → readable CSV | Required bridge between upload and pandas |
| `@st.cache_data` with args | Caches based on file hash | Prevents re-parsing on every widget click |
| `df.select_dtypes()` | Auto-detects numeric vs categorical cols | Makes EDA dynamic — no hardcoded column names |
| `st.download_button()` | Lets user download processed file | Closes the analysis loop — client takes results away |
| `st.stop()` | Halts app execution | Clean null-file handling — shows message, stops |

---

### The file upload pattern (memorise this)

```python
import io

uploaded = st.file_uploader("Upload CSV", type=["csv"])

if uploaded is None:
    st.info("📂 Upload a CSV file to begin analysis.")
    st.stop()               # <── stops execution here; nothing below runs

@st.cache_data
def load_data(file_bytes):
    return pd.read_csv(io.BytesIO(file_bytes))

df = load_data(uploaded.read())
```

**Key rule:** `st.cache_data` receives `file_bytes` (bytes), not the uploaded object itself.  
The uploaded object changes reference on every rerun. Bytes are stable — cache works correctly.

---

### Auto EDA pattern

```python
num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()
```

These two lines replace 20 lines of hardcoded column names. Any CSV the client uploads gets correct analysis automatically.

---

### Download button pattern

```python
csv_bytes = df.to_csv(index=False).encode('utf-8')

st.download_button(
    label     = "⬇️ Download Processed Data",
    data      = csv_bytes,
    file_name = "processed_data.csv",
    mime      = "text/csv",
)
```

Always `.encode('utf-8')` — `download_button` requires bytes, not a string.

---

### Interview answer (Day 143)
> *"My Streamlit apps use `st.file_uploader` with `io.BytesIO` and `@st.cache_data` so the client uploads their own CSV and gets instant EDA — shape, types, nulls, stats, charts, and a downloadable processed file. The cache decorator ensures the CSV is parsed only once regardless of how many widgets the user interacts with. This pattern is the difference between a dashboard and a product."*


---
## ✏️ Section 3 — Practice Tasks

Create a file `day143_app.py`. Each task adds to the same app.  
Test by running: `streamlit run day143_app.py`  
Upload `freelancehub_india.csv` when prompted.

---

### T1 — File Uploader + Null State (10 pts)

Set up the app skeleton with file upload. Handle the case where no file is uploaded.

**Requirements:**
1. `st.set_page_config(page_title="CSV Analyser", layout="wide")`
2. Title: `"📂 Universal CSV Analyser"`, caption: `"Upload any CSV for instant EDA"`
3. Sidebar: `st.file_uploader("Upload your CSV", type=["csv"])`
4. Sidebar subheader: `"Upload Settings"`
5. If no file: show `st.info("📂 Upload a CSV file in the sidebar to begin.")` then call `st.stop()`
6. If file uploaded: show `st.success(f"✅ Loaded: {uploaded.name}")` in the sidebar

**Expected:** App shows the info message when no file is uploaded. After upload, shows success with filename.

---

### T2 — Load + Cache File (10 pts)

Read the uploaded file and display basic shape info.

**Requirements:**
1. Define `load_data(file_bytes)` decorated with `@st.cache_data`
2. Inside: `return pd.read_csv(io.BytesIO(file_bytes))`
3. Call it: `df = load_data(uploaded.read())`
4. Display 3 metric cards in one row: **Rows**, **Columns**, **Total Cells** (rows × cols)
5. Display `st.dataframe(df.head(5), use_container_width=True)` under the subheader `"📋 Data Preview (first 5 rows)"`

**Expected values (FreelanceHub):** Rows=500, Columns=11, Total Cells=5,500

---

### T3 — Auto EDA Section (15 pts)

Build an automatic EDA block that works on any uploaded CSV.

**Requirements:**
1. Subheader: `"🔍 Automatic EDA"`
2. Two columns (equal width):
   - Left: `"📊 Column Types"` — a dataframe showing each column name + dtype (use `df.dtypes.reset_index()` with columns `['Column','Type']`)
   - Right: `"🚨 Null Analysis"` — a dataframe showing each column + null count + null % (rounded to 1 decimal)
3. Below, full-width: `"📈 Descriptive Statistics"` — `st.dataframe(df.describe().T.round(2), use_container_width=True)`
4. Below stats, show two `st.metric` side by side:
   - `"Numeric Columns"` → count of numeric columns
   - `"Categorical Columns"` → count of object/string columns

**Expected values (FreelanceHub):** 5 numeric cols, 6 categorical cols, 0 nulls in every column

---

### T4 — Dynamic Filter (15 pts)

Build a filter section using auto-detected categorical columns.

**Requirements:**
1. Subheader: `"🎯 Dynamic Filter"`
2. In the **sidebar** (below file uploader):
   - `st.sidebar.subheader("Filter Data")`
   - `st.sidebar.selectbox("Filter by column:", options=cat_cols)` where `cat_cols` is detected via `select_dtypes`
   - `st.sidebar.multiselect("Select values:", options=sorted(df[selected_col].unique()), default=sorted(df[selected_col].unique()))`
3. Apply filter: `df_filtered = df[df[selected_col].isin(selected_vals)]`
4. Show two metrics after filtering:
   - `"Filtered Rows"` with delta = `f"{len(df_filtered) - len(df)} rows"`
   - `"Avg Budget (₹)"` — mean of `budget_inr` in filtered df, formatted to 0 decimal

**Expected (status=Completed filter):** Filtered Rows=316, Avg Budget=₹79,388

---

### T5 — Distribution Chart (10 pts)

Build a dynamic histogram for any numeric column.

**Requirements:**
1. Subheader: `"📊 Distribution Analysis"`
2. `st.selectbox("Select numeric column:", options=num_cols)` — inside the **main area** (not sidebar)
3. Plotly histogram: `px.histogram(df_filtered, x=selected_num_col, nbins=20, title=f"Distribution of {selected_num_col}")`
4. Set `color_discrete_sequence=['#1F3864']`
5. `st.plotly_chart(fig, use_container_width=True)`
6. Below chart: show `st.caption` with: `f"Mean: ₹{df_filtered[selected_num_col].mean():,.0f} | Median: ₹{df_filtered[selected_num_col].median():,.0f} | Std: ₹{df_filtered[selected_num_col].std():,.0f}"`  
   *(Use the ₹ symbol for budget_inr — acceptable for any column in this dataset)*

**Expected (budget_inr, all data):** Mean: ₹79,030 | Median: ₹78,519 | Std: ₹42,982

---

### T6 — Processed Download (10 pts)

Add derived columns and a download button.

**Requirements:**
1. Add column `budget_tier` using `pd.cut`:
   - Bins: `[0, 25000, 75000, 150001]`
   - Labels: `['Low (<25k)', 'Mid (25-75k)', 'High (>75k)']`
   - Apply to `df_filtered` (not df_raw — user downloads what they filtered)
2. Show `st.dataframe(df_filtered[['project_id','category','budget_inr','net_earnings','budget_tier']].head(10), use_container_width=True)` under subheader `"⬇️ Processed Data Preview"`
3. Convert to CSV bytes: `csv_bytes = df_filtered.to_csv(index=False).encode('utf-8')`
4. `st.download_button(label="⬇️ Download Processed CSV", data=csv_bytes, file_name="processed_freelancehub.csv", mime="text/csv")`
5. Caption below button: `f"Downloading {len(df_filtered)} rows × {len(df_filtered.columns)} columns"`

**Expected (all data):** 500 rows × 12 columns (11 original + budget_tier)

---

### T7 — NRA Insight (10 pts)

Write a dynamic NRA insight in `st.success()`.

**Requirements:**
1. Compute avg `net_earnings` by `freelancer_level` from `df_filtered`
2. Find the **highest earning level** and its average
3. Find the **lowest earning level** and its average
4. Display in `st.success()` using NRA format:
   - **Number:** Name highest level + exact ₹ value + lowest level + exact ₹ value + gap
   - **Reason:** Explain why this gap might exist (market positioning, project complexity)
   - **Action:** Specific platform, specific target level, time-bound step

**Expected values (all data):**
- Highest: Level 1 → ₹69,799
- Lowest: Top Rated → ₹63,623
- Gap: ₹6,176 (9.7% higher for Level 1)

*(Counterintuitive finding — Level 1 earns more net than Top Rated. Your NRA must explain this, not ignore it.)*

---

### ★ Bonus — Correlation Heatmap (10★)

**Requirements:**
1. Subheader: `"🔗 Correlation Heatmap (Numeric Columns)"`
2. Compute `df_filtered[num_cols].corr().round(3)`
3. Plotly `imshow`: `px.imshow(corr_matrix, text_auto=True, color_continuous_scale='RdBu_r', zmin=-1, zmax=1)`
4. Set title: `"Pearson Correlation — Numeric Features"`
5. `st.plotly_chart(fig, use_container_width=True)`
6. Below: `st.caption` identifying the strongest non-self correlation pair and its value

**Expected:** `budget_inr ↔ net_earnings = 0.997` (strongest pair — almost perfect, since net = budget × fee multiplier)

---


---
## 🔑 Section 4 — Answer Key

Complete `day143_app.py`. Study line by line — every pattern here appears in real Upwork deliverables.


---
## 📊 Section 5 — Scoring Rubric

| Task | Topic | Points | Criteria |
|---|---|---|---|
| **T1** | File uploader + null state | 10 | `st.file_uploader` in sidebar ✓ · `st.stop()` on null ✓ · success message with filename ✓ |
| **T2** | Load + cache | 10 | `@st.cache_data` correct ✓ · `io.BytesIO` used ✓ · 3 metric cards exact values ✓ · head(5) displayed ✓ |
| **T3** | Auto EDA | 15 | dtype table ✓ · null table with % ✓ · describe().T ✓ · 2 metrics (5 num, 6 cat) ✓ |
| **T4** | Dynamic filter | 15 | sidebar selectbox + multiselect ✓ · filter applied ✓ · Filtered Rows=316 ✓ · Avg Budget=₹79,388 ✓ |
| **T5** | Histogram | 10 | `nbins=20` ✓ · dark blue `#1F3864` ✓ · caption with mean/median/std exact ✓ |
| **T6** | Download button | 10 | `budget_tier` derived correctly ✓ · `.encode('utf-8')` ✓ · caption with row×col count ✓ |
| **T7** | NRA insight | 10 | Level 1 ₹69,799 cited ✓ · Top Rated ₹63,623 cited ✓ · gap ₹6,176 ✓ · platform fee explanation ✓ · time-bound action ✓ |
| **★ Bonus** | Correlation heatmap | 10★ | `px.imshow` ✓ · `text_auto=True` ✓ · `RdBu_r` scale ✓ · strongest pair caption (0.997) ✓ |
| **Total** | | **80 + 10★** | |

---

### Deduction guide

| Error type | Deduction |
|---|---|
| Used `uploaded.read()` without `io.BytesIO` (direct StringIO from bytes) | −3 |
| `@st.cache_data` missing or placed incorrectly | −4 |
| `st.stop()` missing — app crashes instead of showing message | −3 |
| Filter not applied to `df_filtered` (still using `df`) | −4 |
| budget_tier bins wrong (e.g. last bin ≤150000 not 150001) | −2 |
| `.encode('utf-8')` missing from download button | −3 |
| NRA missing the platform-fee explanation for the counterintuitive gap | −4 |
| Bonus: `zmin/zmax` not set (colour scale not normalised to −1,1) | −2 |

---

### Month 8 Scorecard

| Day | Topic | Score |
|---|---|---|
| 141 | Streamlit Fundamentals | 80/80 + 10★ |
| 142 | Session State + Tabs + Forms | 80/80 + 10★ |
| **143** | **File Upload + Dynamic EDA** | **— / 80 + 10★** |
